# MA3632 — Workshop 6: Linear and Polynomial Regression

This workshop accompanies Lecture 6. We implement OLS from scratch using the normal
equations, explore the geometric interpretation via projection, fit polynomial models
and observe overfitting, then apply Ridge and Lasso regularisation with
cross-validation to select the regularisation parameter.

Work through all parts in order. Take-home exercises are at the end.

---

## Part A — OLS from scratch: the normal equations

We implement OLS directly from the formula $\hat{\mathbf{w}} = (X^\top X)^{-1} X^\top
\mathbf{y}$ and verify the results against a worked example from the lecture.

### A1. Imports and helper functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.datasets import fetch_california_housing
import warnings
warnings.filterwarnings("ignore")

def add_intercept(X):
    """Prepend a column of ones to X to absorb the intercept."""
    n = X.shape[0]
    return np.hstack([np.ones((n, 1)), X])

def ols_fit(X, y):
    """
    OLS via the normal equations.
    X should already include the intercept column.
    Returns w_hat = (X'X)^{-1} X'y.
    """
    return np.linalg.solve(X.T @ X, X.T @ y)

def ols_predict(X, w):
    """Predicted values for design matrix X and coefficient vector w."""
    return X @ w

### A2. Reproducing the lecture exercise

In [ ]:
# Four observations from the lecture exercise
x = np.array([0., 1., 2., 3.])
y = np.array([1., 3., 2., 5.])

X = add_intercept(x.reshape(-1, 1))

print("Design matrix X:")
print(X)
print()

XtX = X.T @ X
Xty = X.T @ y
print("X'X ="); print(XtX)
print()
print("X'y =", Xty)

w_hat = ols_fit(X, y)
print()
print(f"OLS solution: w0 = {w_hat[0]:.4f}, w1 = {w_hat[1]:.4f}")
print(f"Fitted line:  f(x) = {w_hat[0]:.4f} + {w_hat[1]:.4f} * x")

y_hat = ols_predict(X, w_hat)
residuals = y - y_hat
print()
print("Fitted values:", np.round(y_hat, 4))
print("Residuals:    ", np.round(residuals, 4))
print(f"Sum of residuals:          {residuals.sum():.10f}  (should be ~0)")
print(f"Residuals . x values:      {(x * residuals).sum():.10f}  (should be ~0)")

The residuals sum to zero and are orthogonal to the predictor values, confirming the
geometric property proved in the lecture: the residual vector is orthogonal to every
column of $X$.

### A3. The hat matrix and projection

In [ ]:
# Hat matrix H = X(X'X)^{-1}X'
H = X @ np.linalg.inv(X.T @ X) @ X.T

print("Hat matrix H (4x4):")
print(np.round(H, 4))
print()

# Verify idempotency: H^2 = H
print(f"Max entry of |H^2 - H|: {np.abs(H @ H - H).max():.2e}  (should be ~0)")

# Verify symmetry
print(f"Max entry of |H - H'|:  {np.abs(H - H.T).max():.2e}  (should be ~0)")

# Fitted values as projection
y_hat_proj = H @ y
print()
print("y_hat via H @ y:", np.round(y_hat_proj, 4))
print("y_hat via X @ w:", np.round(y_hat, 4))
print("(Should be identical)")

The hat matrix is both idempotent ($H^2 = H$) and symmetric ($H^\top = H$), confirming
that it is an orthogonal projection matrix. Applying it twice is the same as applying it
once, because projecting an already-projected vector does nothing.

---
## Part B — Multiple linear regression on real data

We apply OLS to the California Housing dataset: predict median house value from eight
features. This part also computes the full suite of regression metrics from the lecture.

### B1. Loading and splitting the data

In [ ]:
# California Housing — used throughout Parts B-F.
# Falls back to a locally generated synthetic dataset with the same columns
# if the network fetch is unavailable, so the workshop runs offline.
feature_names = [
    "MedInc", "HouseAge", "AveRooms", "AveBedrms",
    "Population", "AveOccup", "Latitude", "Longitude"
]

try:
    raw = fetch_california_housing(as_frame=True)
    df_housing = raw.frame.copy()
    X_raw = df_housing[feature_names].values
    y_raw = df_housing["MedHouseVal"].values
except Exception as e:
    print(f"California Housing fetch unavailable ({type(e).__name__}); using synthetic fallback.")
    n_synth = 3000
    rng_synth = np.random.default_rng(seed=42)
    med_inc    = rng_synth.gamma(shape=5.0, scale=0.8, size=n_synth)
    house_age  = rng_synth.uniform(1, 52, size=n_synth)
    ave_rooms  = rng_synth.normal(5.5, 1.2, size=n_synth).clip(min=1)
    ave_bedrms = ave_rooms * rng_synth.uniform(0.15, 0.35, size=n_synth)
    population = rng_synth.gamma(shape=3.0, scale=400, size=n_synth)
    ave_occup  = rng_synth.normal(3.0, 0.7, size=n_synth).clip(min=1)
    latitude   = rng_synth.uniform(32.5, 42.0, size=n_synth)
    longitude  = rng_synth.uniform(-124.3, -114.3, size=n_synth)
    med_house_val = (
        0.5 * med_inc + 0.01 * (52 - house_age) - 0.05 * ave_occup
        + rng_synth.normal(0, 0.4, size=n_synth)
    ).clip(min=0.15, max=5.0)
    X_raw = np.column_stack([
        med_inc, house_age, ave_rooms, ave_bedrms,
        population, ave_occup, latitude, longitude
    ])
    y_raw = med_house_val

print("Features:", feature_names)
print(f"Dataset shape: {X_raw.shape}")
print(f"Target range:  {y_raw.min():.2f} -- {y_raw.max():.2f}  (median house value, $100k)")

X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=0
)

sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc  = sc.transform(X_te)

print(f"\nTraining: {X_tr_sc.shape[0]} samples  |  Test: {X_te_sc.shape[0]} samples")

### B2. Fitting OLS and computing metrics

In [ ]:
def regression_metrics(y_true, y_pred, label=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{label}")
    print(f"  RMSE: {rmse:.4f}   MAE: {mae:.4f}   R²: {r2:.4f}")
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

# OLS via sklearn (uses SVD internally, equivalent to normal equations)
lr = LinearRegression()
lr.fit(X_tr_sc, y_tr)

metrics_train = regression_metrics(y_tr, lr.predict(X_tr_sc), "OLS — training set")
metrics_test  = regression_metrics(y_te, lr.predict(X_te_sc),  "OLS — test set    ")

print()
print("Coefficients (standardised features):")
for name, coef in zip(feature_names, lr.coef_):
    print(f"  {name:<20} {coef:+.4f}")
print(f"  Intercept             {lr.intercept_:+.4f}")

### B3. Residual diagnostics

In [ ]:
residuals_te = y_te - lr.predict(X_te_sc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs fitted values
axes[0].scatter(lr.predict(X_te_sc), residuals_te, s=5, alpha=0.3, color="steelblue")
axes[0].axhline(0, color="firebrick", lw=1)
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs fitted (test set)")

# Histogram of residuals
axes[1].hist(residuals_te, bins=60, color="steelblue", edgecolor="white", lw=0.3)
axes[1].axvline(0, color="firebrick", lw=1)
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution (test set)")

plt.tight_layout()
plt.show()

A well-specified linear model should produce residuals that are approximately symmetric
around zero with no pattern against fitted values. Systematic curvature or fanning
(heteroscedasticity) in the left plot signals model misspecification.

---
## Part C — Polynomial regression and overfitting

We fit polynomial models of increasing degree to a small synthetic dataset and observe
the transition from underfitting to overfitting.

### C1. Synthetic data

In [ ]:
rng = np.random.default_rng(3)
n_tr = 20
n_te = 200

x_cos_tr = np.sort(rng.uniform(0, 2, n_tr))
y_cos_tr = np.cos(np.pi * x_cos_tr / 2) + rng.normal(0, 0.15, n_tr)

x_cos_te = np.linspace(0, 2, n_te)
y_cos_te = np.cos(np.pi * x_cos_te / 2)   # noiseless true function

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_cos_tr, y_cos_tr, s=40, color="steelblue", zorder=3, label="Training data")
ax.plot(x_cos_te, y_cos_te, color="black", lw=1.5, label="True function")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Synthetic data: y = cos(pi*x/2) + noise")
ax.legend()
plt.tight_layout()
plt.show()

### C2. Fitting polynomials of increasing degree

In [ ]:
degrees = [1, 2, 3, 5, 9, 14]
train_rmse = []
test_rmse  = []

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, d in enumerate(degrees):
    pipe = Pipeline([
        ("poly",  PolynomialFeatures(degree=d, include_bias=False)),
        ("scale", StandardScaler()),
        ("ols",   LinearRegression())
    ])
    pipe.fit(x_cos_tr.reshape(-1, 1), y_cos_tr)

    y_tr_pred = pipe.predict(x_cos_tr.reshape(-1, 1))
    y_te_pred = pipe.predict(x_cos_te.reshape(-1, 1))

    tr_rmse = np.sqrt(mean_squared_error(y_cos_tr, y_tr_pred))
    te_rmse = np.sqrt(mean_squared_error(y_cos_te, y_te_pred))
    train_rmse.append(tr_rmse)
    test_rmse.append(te_rmse)

    ax = axes[i]
    ax.scatter(x_cos_tr, y_cos_tr, s=20, color="steelblue", zorder=3, alpha=0.8)
    ax.plot(x_cos_te, y_cos_te,      color="black",     lw=1.2, label="True")
    ax.plot(x_cos_te, y_te_pred, color="firebrick", lw=1.2, label=f"d={d}")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"degree = {d}  |  train RMSE = {tr_rmse:.3f}  |  test RMSE = {te_rmse:.3f}",
                 fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlabel("x")

plt.suptitle("Polynomial regression: underfitting to overfitting", y=1.01)
plt.tight_layout()
plt.show()

### C3. Training vs test RMSE as a function of degree

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(degrees, train_rmse, marker="o", color="steelblue", label="Training RMSE")
ax.plot(degrees, test_rmse,  marker="o", color="firebrick",  label="Test RMSE")
ax.set_xlabel("Polynomial degree")
ax.set_ylabel("RMSE")
ax.set_title("Bias–variance trade-off: training vs test RMSE")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'Degree':>7}  {'Train RMSE':>12}  {'Test RMSE':>12}")
for d, tr, te in zip(degrees, train_rmse, test_rmse):
    print(f"{d:>7}  {tr:>12.4f}  {te:>12.4f}")

Training RMSE decreases monotonically with degree — a higher-degree polynomial can
always fit the training points at least as well. Test RMSE follows a U-shape: it
falls as degree increases from 1 to the optimal value, then rises as the model begins
to overfit. The degree-9 and degree-14 fits oscillate wildly (Runge phenomenon) while
achieving near-zero training error.

---
## Part D — Ridge regression

We apply Ridge to the California Housing data and study the effect of the regularisation
parameter $\lambda$ on the coefficient estimates and test error.

### D1. Ridge coefficient paths

In [ ]:
lambdas = np.logspace(-3, 3, 100)
coef_paths = []

for lam in lambdas:
    ridge = Ridge(alpha=lam)
    ridge.fit(X_tr_sc, y_tr)
    coef_paths.append(ridge.coef_.copy())

coef_paths = np.array(coef_paths)   # shape (100, 8)

fig, ax = plt.subplots(figsize=(10, 5))
for j, name in enumerate(feature_names):
    ax.plot(lambdas, coef_paths[:, j], lw=1.5, label=name)

ax.set_xscale("log")
ax.set_xlabel("lambda (regularisation parameter)")
ax.set_ylabel("Coefficient value")
ax.set_title("Ridge coefficient paths (California Housing)")
ax.axvline(1.0, color="black", ls="--", lw=0.8, label="lambda = 1")
ax.legend(fontsize=8, loc="right")
plt.tight_layout()
plt.show()

All coefficients shrink towards zero as $\lambda$ increases. Features with initially
large coefficients shrink faster. As $\lambda \to \infty$, all coefficients approach
zero and the model predicts the training mean everywhere.

### D2. Choosing $\lambda$ by cross-validation

In [ ]:
# RidgeCV searches over a grid of alpha values using efficient LOO-CV
alphas_grid = np.logspace(-3, 3, 200)
ridge_cv = RidgeCV(alphas=alphas_grid, scoring="neg_root_mean_squared_error", cv=10)
ridge_cv.fit(X_tr_sc, y_tr)

print(f"Best lambda (RidgeCV): {ridge_cv.alpha_:.4f}")

# Evaluate on test set
ridge_best = Ridge(alpha=ridge_cv.alpha_)
ridge_best.fit(X_tr_sc, y_tr)
regression_metrics(y_te, ridge_best.predict(X_te_sc), f"Ridge (lambda={ridge_cv.alpha_:.4f}) — test set")
regression_metrics(y_te, lr.predict(X_te_sc),          "OLS (no regularisation)   — test set")

### D3. Manually plotting CV error vs $\lambda$

In [ ]:
kf = KFold(n_splits=10, shuffle=True, random_state=0)
lambdas_plot = np.logspace(-3, 3, 60)
cv_rmse_mean = []
cv_rmse_se   = []

for lam in lambdas_plot:
    scores = -cross_val_score(
        Ridge(alpha=lam), X_tr_sc, y_tr,
        cv=kf, scoring="neg_root_mean_squared_error"
    )
    cv_rmse_mean.append(scores.mean())
    cv_rmse_se.append(scores.std() / np.sqrt(10))

cv_rmse_mean = np.array(cv_rmse_mean)
cv_rmse_se   = np.array(cv_rmse_se)

best_idx  = cv_rmse_mean.argmin()
threshold = cv_rmse_mean[best_idx] + cv_rmse_se[best_idx]
qualifying = lambdas_plot[cv_rmse_mean <= threshold]
lam_1se   = qualifying.max()   # largest lambda within 1 SE (simpler/more regularised)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lambdas_plot, cv_rmse_mean, color="steelblue", lw=1.5, label="CV RMSE")
ax.fill_between(lambdas_plot,
                cv_rmse_mean - cv_rmse_se,
                cv_rmse_mean + cv_rmse_se,
                alpha=0.2, color="steelblue", label="±1 SE")
ax.axvline(lambdas_plot[best_idx], color="firebrick", ls="--", lw=1,
           label=f"Min at lambda = {lambdas_plot[best_idx]:.3f}")
ax.axvline(lam_1se, color="darkorange", ls="--", lw=1,
           label=f"1-SE rule (lambda = {lam_1se:.3f})")
ax.set_xscale("log")
ax.set_xlabel("lambda")
ax.set_ylabel("CV RMSE")
ax.set_title("Ridge: 10-fold CV error vs lambda (California Housing)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Exact minimiser:    lambda = {lambdas_plot[best_idx]:.4f}  CV RMSE = {cv_rmse_mean[best_idx]:.4f}")
print(f"1-SE rule selects:  lambda = {lam_1se:.4f}")

---
## Part E — Lasso regression and variable selection

Lasso applies an $\ell_1$ penalty that drives some coefficients to exactly zero,
performing simultaneous shrinkage and variable selection.

### E1. Lasso coefficient paths

In [ ]:
from sklearn.linear_model import lasso_path

# lasso_path returns the entire regularisation path efficiently
alphas_lasso, coef_lasso_path, _ = lasso_path(X_tr_sc, y_tr, eps=1e-4, n_alphas=200)

fig, ax = plt.subplots(figsize=(10, 5))
for j, name in enumerate(feature_names):
    ax.plot(alphas_lasso, coef_lasso_path[j], lw=1.5, label=name)

ax.set_xscale("log")
ax.invert_xaxis()   # convention: plot from high lambda (all zero) to low lambda
ax.set_xlabel("lambda (decreasing ->)")
ax.set_ylabel("Coefficient value")
ax.set_title("Lasso coefficient paths (California Housing)")
ax.legend(fontsize=8, loc="right")
plt.tight_layout()
plt.show()

Unlike Ridge, Lasso coefficients reach exactly zero at finite $\lambda$ and stay zero
for all larger values. Features enter the model sequentially as $\lambda$ decreases,
with the most important features entering first.

### E2. Choosing $\lambda$ for Lasso by cross-validation

In [ ]:
lasso_cv = LassoCV(cv=10, random_state=0, max_iter=5000, n_alphas=200)
lasso_cv.fit(X_tr_sc, y_tr)

print(f"Best lambda (LassoCV): {lasso_cv.alpha_:.6f}")
print()

lasso_best = Lasso(alpha=lasso_cv.alpha_, max_iter=5000)
lasso_best.fit(X_tr_sc, y_tr)

print("Lasso coefficients at selected lambda:")
for name, coef in zip(feature_names, lasso_best.coef_):
    status = "  (zeroed out)" if coef == 0 else ""
    print(f"  {name:<20} {coef:+.4f}{status}")

print()
regression_metrics(y_te, lasso_best.predict(X_te_sc), "Lasso (CV lambda) — test set")

### E3. Comparing OLS, Ridge, and Lasso

In [ ]:
models = {
    "OLS":   lr,
    "Ridge": ridge_best,
    "Lasso": lasso_best,
}

print(f"{'Model':<10}  {'RMSE':>8}  {'MAE':>8}  {'R²':>8}")
print("-" * 40)
for name, model in models.items():
    y_pred = model.predict(X_te_sc)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    print(f"{name:<10}  {rmse:>8.4f}  {mae:>8.4f}  {r2:>8.4f}")

# Coefficient comparison plot
fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(len(feature_names))
width = 0.25

ax.bar(x_pos - width, lr.coef_,          width, label="OLS",   color="steelblue",  alpha=0.8)
ax.bar(x_pos,         ridge_best.coef_,  width, label="Ridge", color="firebrick",  alpha=0.8)
ax.bar(x_pos + width, lasso_best.coef_,  width, label="Lasso", color="darkorange", alpha=0.8)

ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Coefficient value")
ax.set_title("Coefficient comparison: OLS vs Ridge vs Lasso (standardised features)")
ax.axhline(0, color="black", lw=0.8)
ax.legend()
plt.tight_layout()
plt.show()

---
## Part F — Regularised polynomial regression

We combine polynomial feature expansion with Ridge regularisation: fit a high-degree
polynomial but penalise the coefficients to prevent oscillation.

In [ ]:
# Use the synthetic cosine data from Part C
# Fix a high degree but vary lambda

degree_fixed = 9
lambdas_poly = [0.0001, 0.01, 1.0, 10.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, lam in zip(axes, lambdas_poly):
    if lam == 0.0:
        model = Pipeline([
            ("poly",  PolynomialFeatures(degree=degree_fixed, include_bias=False)),
            ("scale", StandardScaler()),
            ("reg",   LinearRegression())
        ])
    else:
        model = Pipeline([
            ("poly",  PolynomialFeatures(degree=degree_fixed, include_bias=False)),
            ("scale", StandardScaler()),
            ("reg",   Ridge(alpha=lam))
        ])
    model.fit(x_cos_tr.reshape(-1, 1), y_cos_tr)
    y_pred = model.predict(x_cos_te.reshape(-1, 1))
    te_rmse = np.sqrt(mean_squared_error(y_cos_te, y_pred))

    ax.scatter(x_cos_tr, y_cos_tr, s=20, color="steelblue", zorder=3, alpha=0.8)
    ax.plot(x_cos_te, y_cos_te,   color="black",     lw=1.2, label="True")
    ax.plot(x_cos_te, y_pred, color="firebrick", lw=1.2, label=f"lambda={lam}")
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(f"lambda={lam}  |  test RMSE={te_rmse:.3f}", fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel("x")

plt.suptitle(f"Degree-{degree_fixed} polynomial + Ridge: effect of lambda", y=1.02)
plt.tight_layout()
plt.show()

With $\lambda$ very small, the degree-9 polynomial overfits and oscillates.
As $\lambda$ increases, the fit becomes smoother and closer to the true cosine.
Too large a $\lambda$ over-smooths and the fit degrades towards a constant.
Cross-validation selects the $\lambda$ that balances these two extremes.

In [ ]:
# Select lambda by CV for the degree-9 polynomial on the synthetic data
alphas_cv = np.logspace(-4, 2, 80)
cv_rmse_poly = []

kf_poly = KFold(n_splits=5, shuffle=True, random_state=1)

for lam in alphas_cv:
    pipe = Pipeline([
        ("poly",  PolynomialFeatures(degree=degree_fixed, include_bias=False)),
        ("scale", StandardScaler()),
        ("reg",   Ridge(alpha=lam))
    ])
    scores = -cross_val_score(pipe, x_cos_tr.reshape(-1,1), y_cos_tr,
                               cv=kf_poly, scoring="neg_root_mean_squared_error")
    cv_rmse_poly.append(scores.mean())

cv_rmse_poly = np.array(cv_rmse_poly)
best_lam = alphas_cv[cv_rmse_poly.argmin()]

print(f"CV-selected lambda for degree-{degree_fixed} polynomial: {best_lam:.4f}")

pipe_best = Pipeline([
    ("poly",  PolynomialFeatures(degree=degree_fixed, include_bias=False)),
    ("scale", StandardScaler()),
    ("reg",   Ridge(alpha=best_lam))
])
pipe_best.fit(x_cos_tr.reshape(-1, 1), y_cos_tr)
te_rmse_best = np.sqrt(mean_squared_error(
    y_cos_te, pipe_best.predict(x_cos_te.reshape(-1, 1))
))
print(f"Test RMSE with CV-selected lambda: {te_rmse_best:.4f}")

# Compare to best plain polynomial from Part C
best_d_idx = np.argmin(test_rmse)
print(f"Best plain polynomial (degree={degrees[best_d_idx]}): test RMSE = {test_rmse[best_d_idx]:.4f}")

---
## Take-home exercises

**Exercise 1.**
Using the California Housing data from Part B, add squared and interaction terms for
the two most important features (by OLS coefficient magnitude) using
`PolynomialFeatures(degree=2, interaction_only=False)`. Fit OLS on the expanded feature
set and compare test RMSE and $R^2$ to the plain linear model. Does the polynomial
expansion help? Check the residual plot for signs of remaining nonlinearity.

**Exercise 2.**
Implement Ridge regression from scratch using only NumPy: given standardised design
matrix $X$, target vector $\mathbf{y}$, and regularisation parameter $\lambda$, compute
$\hat{\mathbf{w}}_\lambda = (X^\top X + n\lambda I)^{-1} X^\top \mathbf{y}$ directly.
Verify that your implementation matches `sklearn.linear_model.Ridge` for several values
of $\lambda$ on the California Housing data. Note the factor of $n$ in the penalty:
sklearn's `Ridge(alpha=a)` minimises $\|\mathbf{y} - X\mathbf{w}\|^2 + a\|\mathbf{w}\|^2$
(without the $1/n$ factor), so you will need to account for this when comparing.

**Exercise 3.**
Using the synthetic cosine data from Part C, repeat the bias--variance bootstrap
experiment from Workshop~5 (Part C2) but now for Ridge-regularised degree-9 polynomials.
Fix $k = 200$ bootstrap resamples and evaluate at three query points $x \in
\{0.5, 1.0, 1.5\}$. Plot estimated bias$^2$ and variance as functions of $\lambda$ on
a log scale. Confirm that as $\lambda$ increases, variance falls and bias rises, and
identify the $\lambda$ that minimises the total error.

**Exercise 4.**
The elastic net combines Ridge and Lasso penalties:
$\hat{R}(\mathbf{w}) = n^{-1}\|\mathbf{y} - X\mathbf{w}\|^2
+ \lambda[\alpha\|\mathbf{w}\|_1 + (1-\alpha)\|\mathbf{w}\|^2]$,
where $\alpha \in [0,1]$ controls the mix. Use `sklearn.linear_model.ElasticNetCV`
to fit an elastic net on the California Housing data with 10-fold CV over a grid of
$\lambda$ values and $\alpha \in \{0.1, 0.5, 0.9\}$. Compare the selected coefficients
and test RMSE to the Lasso result from Part E. For which value of $\alpha$ does the
elastic net most closely recover the Lasso solution?